# SmolVLM Evaluation on PHLOP Dataset

Zero-shot evaluation (4 prompt scenarios) and fine-tuning (4 difficulty configs).

In [ ]:
%pip install transformers accelerate
%pip install peft bitsandbytes
%pip install datasets huggingface_hub
%pip install decord Pillow matplotlib

## Configuration

In [ ]:
import os
import sys
from pathlib import Path
from huggingface_hub import login

sys.path.insert(0, str(Path(".").resolve()))

from phlop_eval_common import (
    load_phlop_splits, EVAL_OPTIONS,
    FINE_TUNE_CONFIGS, get_val_difficulty_filter,
)
from smol_eval import (
    run_zero_shot_smolvlm,
    run_finetune_smolvlm,
    run_test_comparison_smolvlm,
    SMOLVLM_MODEL_ID,
)

REPO_ID = "zimmari-ai/phlop"
HF_TOKEN = os.environ.get("HF_TOKEN", True)
CAMERA_MODE = "static"
MAX_SAMPLES = 4000
MAX_STEPS = 50
OUTPUT_DIR = "./smolvlm_checkpoints"

In [ ]:
if isinstance(HF_TOKEN, str) and HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)

## Load Dataset from HuggingFace

In [ ]:
splits = load_phlop_splits(REPO_ID, token=HF_TOKEN)
for name, ds in splits.items():
    print(f"  {name}: {len(ds)} scenes")

## Part 1: Zero-shot Evaluation

4 prompt scenarios: no_additional_info, taxonomy_only, physics_only, taxonomy_and_physics

In [ ]:
import json

os.makedirs("results", exist_ok=True)

zero_shot_results = run_zero_shot_smolvlm(
    splits,
    camera_mode=CAMERA_MODE,
    max_samples=MAX_SAMPLES,
)

with open("results/smolvlm_zero_shot.json", "w") as f:
    json.dump(zero_shot_results, f, indent=2, default=str)
print(f"\nSaved zero-shot results to results/smolvlm_zero_shot.json")

## Part 2: Fine-tuning

Four training configurations:
1. **easy** - train on easy questions, validate on medium/hard/very_hard
2. **easy_medium** - train on easy+medium, validate on hard/very_hard
3. **hard** - train on hard questions, validate on easy/medium/very_hard
4. **full** - train on all questions

In [ ]:
print("Fine-tuning configurations:")
for name, cfg in FINE_TUNE_CONFIGS.items():
    val_diff = get_val_difficulty_filter(cfg["train_difficulty"])
    print(f"  {name}: train={cfg['train_difficulty']}, val={val_diff}")

saved_dirs = run_finetune_smolvlm(
    splits,
    output_dir=OUTPUT_DIR,
    config_name=None,  # None = run all configs
    max_steps=MAX_STEPS,
    camera_mode=CAMERA_MODE,
)
print(f"\nSaved checkpoints: {saved_dirs}")

## Part 3: Test Comparison

Compare base model vs all fine-tuned checkpoints on test split.

In [ ]:
model_checkpoints = [("base", SMOLVLM_MODEL_ID)]
for config_name in FINE_TUNE_CONFIGS:
    ckpt = os.path.join(OUTPUT_DIR, config_name)
    if os.path.isdir(ckpt):
        model_checkpoints.append((config_name, ckpt))

print(f"Comparing {len(model_checkpoints)} models on test split...")
comparison = run_test_comparison_smolvlm(
    splits,
    model_checkpoints,
    camera_mode=CAMERA_MODE,
)

comparison_dict = {name: metrics for name, metrics in comparison}
with open("results/smolvlm_comparison.json", "w") as f:
    json.dump(comparison_dict, f, indent=2, default=str)
print(f"\nSaved comparison results to results/smolvlm_comparison.json")

## Results Summary

In [ ]:
print(f"\n{'Model':<20} {'AnswerAcc':>10} {'PhysicsAcc':>10} {'TaxF1':>10}")
print("-" * 55)
for name, metrics in comparison:
    print(f"{name:<20} {metrics['answer_accuracy']:>10.4f} {metrics['physics_signal_accuracy']:>10.4f} {metrics['taxonomy_f1']:>10.4f}")

import csv
with open("results/smolvlm_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "answer_accuracy", "physics_signal_accuracy", "taxonomy_f1"])
    for name, metrics in comparison:
        writer.writerow([name, f"{metrics['answer_accuracy']:.4f}",
                         f"{metrics['physics_signal_accuracy']:.4f}",
                         f"{metrics['taxonomy_f1']:.4f}"])
print(f"\nSaved summary CSV to results/smolvlm_summary.csv")

print(f"\nAll saved artifacts:")
print(f"  Models:  {OUTPUT_DIR}/<easy|easy_medium|hard|full>/")
print(f"  Results: results/smolvlm_zero_shot.json")
print(f"  Results: results/smolvlm_comparison.json")
print(f"  Results: results/smolvlm_summary.csv")